# Tree model comparison

Compare Random Forest and XGBoost with the binary logistic regression results saved in `04_binary_baseline.ipynb`. Use the same seed-42 training and validation CSVs, label mapping (benign = 0, attack = 1), and ordered numerical feature selection. Both trees receive the selected features directly. An attack probability of 0.5 or greater predicts attack.

## Load the saved subsets and identify exact matches

In [1]:
import sys
from pathlib import Path

import numpy as np
import sklearn
import xgboost
from IPython.display import Markdown, display
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from Batch.baseline_data import (
    ATTACK_THRESHOLD,
    TRAINING_PATH,
    VALIDATION_PATH,
    binary_targets,
    load_features,
    select_model_features,
    validation_overlap,
)
from Batch.validate import score_predictions

training, training_labels = load_features(TRAINING_PATH)
validation, validation_labels = load_features(VALIDATION_PATH)
y_train = binary_targets(training_labels, "Training").to_numpy()
y_validation = binary_targets(validation_labels, "Validation").to_numpy()
if (len(training), len(validation)) != (100_000, 25_000):
    raise ValueError("Expected 100,000 training and 25,000 validation rows.")

shared_patterns, matches = validation_overlap(training, validation)
matched_benign, matched_attack = np.bincount(y_validation[matches], minlength=2)
if (shared_patterns, matches.sum(), matched_benign, matched_attack) != (25, 25, 2, 23):
    raise ValueError("Exact overlap differs from the baseline.")
evaluation_masks = {
    "All validation": np.ones(len(validation), dtype=bool),
    "Nonmatching only": ~matches,
}

feature_columns = select_model_features(training)
X_train = training[feature_columns]
X_validation = validation[feature_columns]
print(f"Training: {len(training):,} rows with {len(feature_columns)} selected features")
print(f"Validation: {len(validation):,} rows")
print(f"Exact matches across all 46 original features: {matches.sum()} rows "
      f"({matched_benign} benign, {matched_attack} attack) across {shared_patterns} patterns")
print(f"Nonmatching validation rows: {(~matches).sum():,}")

Training: 100,000 rows with 38 selected features
Validation: 25,000 rows
Exact matches across all 46 original features: 25 rows (2 benign, 23 attack) across 25 patterns
Nonmatching validation rows: 24,975


## Fit the models

Use seed 42, two CPU threads per model, and 100 trees each. Random Forest uses Gini splits with bootstrap sampling, square-root feature sampling and maximum depth 14. XGBoost uses CPU histogram splits, the binary logistic objective, maximum depth 6 and a learning rate of 0.1. Other settings use library defaults. Each model is fit once and scored at the fixed baseline threshold without tuning or early stopping.

Both models use the training-derived balanced weight for class `c`, calculated as `n_train / (2 * n_c)`. Random Forest receives the class-weight mapping and XGBoost receives the corresponding per-row sample weights.

In [2]:
classes = np.array([0, 1])
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes.tolist(), weights.tolist(), strict=True))

forest = RandomForestClassifier(
    n_estimators=100, criterion="gini", max_depth=14, max_features="sqrt",
    bootstrap=True, class_weight=class_weights, random_state=42, n_jobs=2,
)
boosted = XGBClassifier(
    n_estimators=100, objective="binary:logistic", tree_method="hist",
    device="cpu", max_depth=6, learning_rate=0.1, random_state=42, n_jobs=2,
)

forest.fit(X_train, y_train)
boosted.fit(X_train, y_train, sample_weight=weights[y_train])

models = {"Random Forest": forest, "XGBoost": boosted}
predictions = {}
for name, model in models.items():
    attack_index = list(model.classes_).index(1)
    attack_scores = model.predict_proba(X_validation)[:, attack_index]
    predictions[name] = (attack_scores >= ATTACK_THRESHOLD).astype(int)

print(f"scikit-learn {sklearn.__version__}, XGBoost {xgboost.__version__}")
print(f"Training class weights: benign {weights[0]:.6f}, attack {weights[1]:.6f}")

scikit-learn 1.9.1, XGBoost 3.4.1
Training class weights: benign 21.195422, attack 0.512080


## Table 1 — Validation metrics

In [3]:
def show_markdown_table(table):
    header = "| " + " | ".join(table.columns) + " |"
    divider = "| " + " | ".join(["---"] * len(table.columns)) + " |"
    rows = [
        "| " + " | ".join(map(str, row)) + " |"
        for row in table.itertuples(index=False, name=None)
    ]
    display(Markdown("\n".join([header, divider, *rows])))

metrics = score_predictions(y_validation, predictions, evaluation_masks)
count_columns = ["TN", "FP", "FN", "TP"]
rate_columns = [
    "Benign precision", "Benign recall", "Benign F1",
    "Attack precision", "Attack recall", "Attack F1",
    "Balanced accuracy", "Benign false-positive rate",
]
if not metrics[count_columns].sum(axis=1).eq(metrics["Rows"]).all():
    raise ValueError("Confusion matrix counts do not sum to the stated row counts.")

validation_table = metrics.reset_index()
for column in rate_columns:
    validation_table[column] = validation_table[column].map("{:.4%}".format)
show_markdown_table(validation_table)

| Model | Validation | Rows | TN | FP | FN | TP | Benign precision | Benign recall | Benign F1 | Attack precision | Attack recall | Attack F1 | Balanced accuracy | Benign false positives | Actual benign | Benign false-positive rate |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Random Forest | All validation | 25000 | 576 | 9 | 214 | 24201 | 72.9114% | 98.4615% | 83.7818% | 99.9628% | 99.1235% | 99.5414% | 98.7925% | 9 | 585 | 1.5385% |
| Random Forest | Nonmatching only | 24975 | 574 | 9 | 214 | 24178 | 72.8426% | 98.4563% | 83.7345% | 99.9628% | 99.1227% | 99.5410% | 98.7895% | 9 | 583 | 1.5437% |
| XGBoost | All validation | 25000 | 583 | 2 | 122 | 24293 | 82.6950% | 99.6581% | 90.3876% | 99.9918% | 99.5003% | 99.7454% | 99.5792% | 2 | 585 | 0.3419% |
| XGBoost | Nonmatching only | 24975 | 581 | 2 | 122 | 24270 | 82.6458% | 99.6569% | 90.3577% | 99.9918% | 99.4998% | 99.7452% | 99.5784% | 2 | 583 | 0.3431% |

## Table 2 — Change after excluding exact matches

Each change is nonmatching only minus all validation. Score changes use the unrounded metrics.

In [4]:
all_scores = metrics.xs("All validation", level="Validation")
nonmatching_scores = metrics.xs("Nonmatching only", level="Validation")
change_columns = count_columns + rate_columns
changes = nonmatching_scores[change_columns] - all_scores[change_columns]
change_table = changes.copy()
for column in rate_columns:
    change_table[column] = (100 * changes[column]).map("{:+.4f} pp".format)
change_table = change_table.add_prefix("Δ ").reset_index()
show_markdown_table(change_table)

| Model | Δ TN | Δ FP | Δ FN | Δ TP | Δ Benign precision | Δ Benign recall | Δ Benign F1 | Δ Attack precision | Δ Attack recall | Δ Attack F1 | Δ Balanced accuracy | Δ Benign false-positive rate |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Random Forest | -2 | 0 | 0 | -23 | -0.0688 pp | -0.0053 pp | -0.0473 pp | -0.0000 pp | -0.0008 pp | -0.0004 pp | -0.0031 pp | +0.0053 pp |
| XGBoost | -2 | 0 | 0 | -23 | -0.0492 pp | -0.0012 pp | -0.0299 pp | -0.0000 pp | -0.0005 pp | -0.0002 pp | -0.0008 pp | +0.0012 pp |

## Interpretation

The saved logistic regression baseline had 4 benign false alarms and 502 missed attacks on all validation rows, with 98.6301% balanced accuracy. Those error counts stayed at 4 and 502 after excluding the matches, and balanced accuracy moved to 98.6279%.

**Random Forest.** On all validation rows, it had 9 benign false alarms (more than logistic regression) and 214 missed attacks (fewer than logistic regression). Balanced accuracy was 98.7925%. Excluding the 25 matches left 9 false alarms and 214 missed attacks. Balanced accuracy changed by -0.0031 pp and the benign false-positive rate changed by +0.0053 pp.

**XGBoost.** On all validation rows, it had 2 benign false alarms (fewer than logistic regression) and 122 missed attacks (fewer than logistic regression). Balanced accuracy was 99.5792%. Excluding the 25 matches left 2 false alarms and 122 missed attacks. Balanced accuracy changed by -0.0008 pp and the benign false-positive rate changed by +0.0012 pp.